# Nonorthogonal lattice metric geometry

**Prerequisite:** [Nonorthogonal lattice metric geometry](../../../docs/lecturenotes/solidstate/lattices/nonorthogonal-metric-geometry/index.md)

This maintained laboratory uses package-owned direct-lattice and metric geometry to compare fractional and Cartesian descriptions in two and three dimensions. Run the cells from top to bottom. The basis vectors use one consistent illustrative length convention; they are not material parameters.

## Learning objectives

1. Construct direct, reciprocal, covariant, and contravariant basis data.
2. Check direct--reciprocal duality and fundamental-region measure.
3. Show why componentwise rounding can miss the nearest image in a skew cell.
4. Compare metric and Cartesian plane-wave factors in 2D and 3D.
5. Distinguish a geometric metric from an effective-mass or constitutive tensor.

The assertions below provide bounded software and numerical checks. Successful execution is not scientific or pedagogical validation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.periodic.lattice.lattice2d import (
    DirectLattice2D,
    ReciprocalLattice2D,
)
from projectkoios.physkit.periodic.lattice.lattice3d import (
    DirectLattice3D,
    ReciprocalLattice3D,
)
from projectkoios.physkit.periodic.lattice.metric import (
    DirectLatticeMetric,
    NearestLatticeImageResolver,
)

np.set_printoptions(precision=6, suppress=True)

## Two-dimensional oblique cell

For columns of $A$, the geometric metric and reciprocal basis are

$$g=A^{\mathsf T}A, \qquad B=2\pi A^{-\mathsf T}.$$

The package owns these derived arrays and their immutable storage.

In [ ]:
direct_2d = DirectLattice2D(
    a1=np.array([1.0, 0.0]),
    a2=np.array([0.35, 0.90]),
)
reciprocal_2d = ReciprocalLattice2D.from_direct_lattice(direct_2d)
metric_2d = DirectLatticeMetric(direct_2d)

assert np.allclose(
    metric_2d.primitive_basis.T @ metric_2d.reciprocal_basis,
    2.0 * np.pi * np.eye(2),
)
assert np.allclose(metric_2d.reciprocal_basis, reciprocal_2d.primitive_basis)
assert np.allclose(
    metric_2d.reciprocal_metric_tensor,
    (2.0 * np.pi) ** 2 * metric_2d.inverse_metric_tensor,
)
assert np.isclose(metric_2d.measure, direct_2d.measure)

print("direct basis A:\n", metric_2d.primitive_basis)
print("metric g:\n", metric_2d.metric_tensor)
print("inverse metric g^-1:\n", metric_2d.inverse_metric_tensor)
print("cell area:", metric_2d.measure)

## Nearest periodic image

Componentwise rounding is exact for an orthogonal basis away from ties, but it need not minimize Cartesian distance in a skew cell. The package resolver searches the complete finite candidate box implied by the smallest singular value of $A$.

In [ ]:
displacement_fractional = np.array([0.62, 0.57])
naive_translation = np.rint(displacement_fractional).astype(np.int64)
naive_fractional = displacement_fractional - naive_translation
naive_cartesian = metric_2d.to_cartesian(naive_fractional)

nearest = NearestLatticeImageResolver().execute(
    metric_2d,
    displacement_fractional,
)

assert nearest.distance < np.linalg.norm(naive_cartesian)
assert np.array_equal(nearest.translation_indices, np.array([1, 0]))
assert np.allclose(nearest.image_fractional, np.array([-0.38, 0.57]))

print("componentwise-rounded image:", naive_fractional)
print("nearest fractional image:", nearest.image_fractional)
print("nearest Cartesian image:", nearest.image_cartesian)
print("nearest distance:", nearest.distance)

In [ ]:
cell_fractional = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])
cell_cartesian = metric_2d.to_cartesian(cell_fractional)

figure, axis = plt.subplots(figsize=(6.5, 5.2))
axis.plot(cell_cartesian[:, 0], cell_cartesian[:, 1], "o-", label="primitive cell")
axis.quiver(
    [0.0, 0.0],
    [0.0, 0.0],
    [naive_cartesian[0], nearest.image_cartesian[0]],
    [naive_cartesian[1], nearest.image_cartesian[1]],
    angles="xy",
    scale_units="xy",
    scale=1.0,
    color=["tab:red", "tab:green"],
)
axis.scatter(*naive_cartesian, color="tab:red", label="componentwise image")
axis.scatter(
    *nearest.image_cartesian,
    color="tab:green",
    label="nearest image",
)
axis.set(
    xlabel="Cartesian x",
    ylabel="Cartesian y",
    title="Nearest image in an oblique cell",
)
axis.set_aspect("equal")
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
plt.show()

## Metric plane-wave factor and mixed derivative

For a fractional plane wave $\exp(i2\pi\mathbf n\cdot\mathbf s)$,

$$|\mathbf G|^2=(2\pi\mathbf n)^{\mathsf T}g^{-1}(2\pi\mathbf n).$$

A nonzero $g^{12}$ gives the mixed term $2g^{12}\partial_1\partial_2$ in the constant-metric Laplacian.

In [ ]:
mode_2d = np.array([2, -1])
phase_components_2d = 2.0 * np.pi * mode_2d
cartesian_reciprocal_2d = metric_2d.reciprocal_basis @ mode_2d
metric_factor_2d = (
    phase_components_2d
    @ metric_2d.inverse_metric_tensor
    @ phase_components_2d
)
cartesian_factor_2d = cartesian_reciprocal_2d @ cartesian_reciprocal_2d

assert np.isclose(metric_factor_2d, cartesian_factor_2d)
assert not np.isclose(metric_2d.inverse_metric_tensor[0, 1], 0.0)
print("mixed-derivative coefficient 2 g^12:", 2.0 * metric_2d.inverse_metric_tensor[0, 1])
print("plane-wave factor |G|^2:", metric_factor_2d)

## Three-dimensional triclinic cell and Bloch shift

The same package contract applies in three dimensions. For reduced Bloch coordinates $\mathbf q$, $\mathbf k=B\mathbf q$, and the free shifted factor can be evaluated either in Cartesian space or with $g^{-1}$.

In [ ]:
direct_3d = DirectLattice3D(
    a1=np.array([1.0, 0.0, 0.0]),
    a2=np.array([0.20, 0.90, 0.0]),
    a3=np.array([0.10, 0.25, 0.80]),
)
reciprocal_3d = ReciprocalLattice3D.from_direct_lattice(direct_3d)
metric_3d = DirectLatticeMetric(direct_3d)

mode_3d = np.array([1, -2, 1])
reduced_bloch = np.array([0.13, -0.08, 0.17])
phase_components_3d = 2.0 * np.pi * (mode_3d + reduced_bloch)
cartesian_wavevector_3d = metric_3d.reciprocal_basis @ (mode_3d + reduced_bloch)
metric_factor_3d = (
    phase_components_3d
    @ metric_3d.inverse_metric_tensor
    @ phase_components_3d
)
cartesian_factor_3d = cartesian_wavevector_3d @ cartesian_wavevector_3d

assert np.allclose(
    metric_3d.reciprocal_basis,
    reciprocal_3d.primitive_basis_matrix,
)
assert np.isclose(metric_factor_3d, cartesian_factor_3d)
assert np.isclose(metric_3d.measure, direct_3d.measure)

nearest_3d = NearestLatticeImageResolver().execute(
    metric_3d,
    np.array([0.58, 0.56, -0.52]),
)
print("3D metric:\n", metric_3d.metric_tensor)
print("3D cell volume:", metric_3d.measure)
print("shifted free factor |G+k|^2:", metric_factor_3d)
print("3D nearest fractional image:", nearest_3d.image_fractional)

In [ ]:
figure = plt.figure(figsize=(7.0, 5.8))
axis = figure.add_subplot(projection="3d")
colors = ("tab:blue", "tab:orange", "tab:green")
labels = (r"$\mathbf{a}_1$", r"$\mathbf{a}_2$", r"$\mathbf{a}_3$")
for vector, color, label in zip(
    metric_3d.primitive_basis.T,
    colors,
    labels,
    strict=True,
):
    axis.quiver(0.0, 0.0, 0.0, *vector, color=color, label=label)
axis.set(
    xlabel="Cartesian x",
    ylabel="Cartesian y",
    zlabel="Cartesian z",
    title="Triclinic direct-lattice basis",
)
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation, limitations, and exercises

The 2D and 3D calculations verify one constant-basis representation: direct--reciprocal duality, metric identities, exact nearest-image resolution, and equivalent Cartesian and fractional plane-wave factors. The metric is geometric rather than constitutive. These synthetic cells do not represent a named material, and one execution does not establish broad numerical robustness, scientific validation, uncertainty quantification, or pedagogical validation.

The nearest-image search is complete, but its candidate count can grow for a severely ill-conditioned basis. The Laplacian expression is valid for a constant lattice basis; a position-dependent curvilinear metric requires the full Laplace--Beltrami divergence form.

### Exercises

1. Replace the 2D basis with an orthogonal rectangle and verify that the mixed-derivative coefficient vanishes.
2. Vary the 2D skew continuously and identify when componentwise rounding first differs from the nearest image for the selected displacement.
3. Choose another 3D reciprocal mode and reduced Bloch coordinate and repeat the metric/Cartesian factor check.
4. Exchange two 3D primitive vectors. Determine which reported quantities change and which remain invariant.
5. Explain why inserting an effective-mass tensor into the kinetic energy is a new model rather than another name for the geometric metric.